# 02 — The address: `monad_word_coords` and `generate_zeros`
C source: `monad.c` lines 146–190 (commit `eb28ce8`). Port: `tools/monad_py.py: word_coords, generate_zeros`. Ported **as-is**, defects included.

```c
uint64_t v = 0;
for (p = surface; *p; p++) { ci = (c >= 32 && c < 127) ? c - 32 + 1 : 0;  v = v * 95ULL + ci; }
if (v > 0) v--;
double seed = fmod((double)v * MONAD_PHI, 1.0);
*idx = (int)(seed * N);               *E = D_STAR + seed * (OMEGA_ZS - D_STAR);
```

In [1]:
import os, sys, json, math, collections, re, hashlib
import numpy as np
sys.path.insert(0, os.path.abspath("../tools"))
import ptol_bin, monad_py as mp
ROOT  = os.path.abspath("..")
STORE = os.path.join(ROOT, "dirty", "monad_wordnet_v4_20260519.bin")
CBIN  = os.path.join(ROOT, "build", "PtolC_eb28ce8", "ptolemy")
SPACES = "[\u200b\u2009\u2008\u2007\u2006\u2005\u2004\u2003]"   # the eight Fermat separators

## Step by step for `holcus`

In [2]:
s = "holcus"; v = 0
for c in s.encode():
    ci = c-32+1; v = v*95 + ci; print(f"  {chr(c)}  digit {ci:3d}  v = {v}")
v -= 1
x = float(v) * mp.PHI
print("v-1 =", v, "  (double)v*phi =", repr(x), "  fmod ->", repr(math.fmod(x,1.0)))
print("word_coords ->", mp.word_coords(s), "   d* + seed*(Omega-d*) with seed =", math.fmod(x,1.0))

  h  digit  73  v = 73
  o  digit  80  v = 7015
  l  digit  77  v = 666502
  c  digit  68  v = 63317758
  u  digit  86  v = 6015187096
  s  digit  84  v = 571442774204
v-1 = 571442774203   (double)v*phi = 924613831285.9856   fmod -> 0.985595703125
word_coords -> (24639, 0.5625142041015625)    d* + seed*(Omega-d*) with seed = 0.985595703125


## The port reproduces the store exactly
Every one of the 24 485 displayed words, re-addressed from its spelling, lands on the slot and energy the store holds.

In [3]:
m = mp.Monad.from_bin(STORE)
bad = sum(1 for i,v in m.vocab.items() if mp.word_coords(v["word"])[0]!=i or abs(mp.word_coords(v["word"])[1]-v["E"])>1e-12)
print("mismatches:", bad, "of", len(m.vocab))

mismatches: 0 of 24485


## Defect 1 — the seed loses its fractional bits as words get longer
`(double)v * φ` is a 53-bit float. Once the product exceeds 2⁵² the spacing between doubles is ≥ 1, so `fmod(x, 1.0)` is **exactly 0**. Before that the number
of usable fractional bits shrinks by one per factor of two. Result: the number of *distinct* slots a word length can reach falls from thousands to one.

In [4]:
from nltk.corpus import wordnet as wn
lem = sorted({l.lower() for s in wn.all_synsets() for l in s.lemma_names() if l.isalpha() and l.isascii()})
by = collections.defaultdict(list)
for w in lem:
    if mp.token_accept(w): by[len(w)].append(w)
rows = []
for n in range(2,16):
    ws = by[n]; ids = [mp.word_coords(w)[0] for w in ws]
    rows.append((n, len(ws), len(set(ids)), sum(1 for i in ids if i==0)))
print("len  lemmas  distinct-slots  at-slot-0")
for r in rows: print(f"{r[0]:3d} {r[1]:7d} {r[2]:14d} {r[3]:10d}")

len  lemmas  distinct-slots  at-slot-0
  2     132            132          0
  3    1016           1006          0
  4    3040           2862          0
  5    5347           4819          0
  6    8335           5271          0
  7    9983             64        155
  8   11117              1      11117
  9   10913              1      10913
 10    8989              1       8989
 11    6729              2       6728
 12    4544              2       4543
 13    2958              1       2958
 14    1712              1       1712
 15     965              1        965


In [5]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,3.4))
ax.semilogy([r[0] for r in rows], [r[2] for r in rows], "o-"); ax.set_xlabel("word length (letters)"); ax.set_ylabel("distinct slots reached")
ax.axhline(25000, ls=":", c="grey"); ax.set_title("monad_word_coords: address resolution by word length"); fig.tight_layout()
fig.savefig(os.path.join(ROOT,"figures","address_resolution_by_length.png"), dpi=160); print("saved")

/home/rendier/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


saved


Every word of 8 letters or more addresses to **z#0 with E = d\*** (11 117 of 11 117 eight-letter lemmas, 10 913 of 10 913 nine-letter, …). That includes `philadelphos`.
The 2026-05-17 baseline assessment reports `philadelphos` as its "deepest word, zero #0"; this is the arithmetic that put it there.
A second, smaller effect: for lengths 6 and 7 the seed is quantised (6 letters: 5 271 distinct seeds; 7 letters: 64).

In [6]:
for w in ["philadelphos","ptolemy","holcus","seemy","carre"]:
    print(f"{w:13} -> {mp.word_coords(w)}")
print("slot 0 on the store displays:", m.vocab[0]["word"])

philadelphos  -> (0, 0.246)
ptolemy       -> (10156, 0.376463125)
holcus        -> (24639, 0.5625142041015625)
seemy         -> (0, 0.24600796283721924)
carre         -> (1, 0.24601653820037842)
slot 0 on the store displays: seemy


## Defect 2 — `uint64` wraps at ~10 letters
`95¹⁰ ≈ 6·10¹⁹ > 2⁶⁴`. In C the accumulator wraps silently; the port masks to 64 bits to match. This is moot in practice because defect 1 has already zeroed the seed for such words,
but it is a second reason a long word's address cannot be trusted.

## What a non-ASCII word does
Every byte outside 32..126 contributes digit **0**, so a non-ASCII string (UTF-8 bytes) is a run of zero digits:

In [7]:
for w in ["tree","arbre","baum","木","δένδρο","شجرة","ağaç"]:
    print(f"{w!r:10} bytes={list(w.encode())[:8]}  word_coords={mp.word_coords(w)}")

'tree'     bytes=[116, 114, 101, 101]  word_coords=(3109, 0.28594041907876727)
'arbre'    bytes=[97, 114, 98, 114, 101]  word_coords=(18275, 0.4807560792160034)
'baum'     bytes=[98, 97, 117, 109]  word_coords=(14392, 0.4308808291518688)
'木'        bytes=[230, 156, 168]  word_coords=(0, 0.246)
'δένδρο'   bytes=[206, 180, 206, 173, 206, 189, 206, 180]  word_coords=(0, 0.246)
'شجرة'     bytes=[216, 180, 216, 172, 216, 177, 216, 169]  word_coords=(0, 0.246)
'ağaç'     bytes=[97, 196, 159, 97, 195, 167]  word_coords=(17193, 0.46686215332031245)


`木` → three bytes, all digit 0 → `v = 0` → z#0. Any word made only of non-ASCII bytes lands on z#0 with E = d\*; mixed words (`ağaç`) lose the non-ASCII letters.
In the C engine this path is never reached from text (the tokenizer discards those bytes first — notebook 03), only from a direct call.

## The zeros themselves
`generate_zeros` tabulates the first 20 and *computes* the rest with Newton's method on the Riemann–von Mangoldt count (`N(t) = n`). They are approximations, not tabulated zeros.

In [8]:
import mpmath
print("  n     true zero    monad z[n-1]     error")
for n in [21, 100, 1000, 5000, 25000]:
    z = float(mpmath.zetazero(n).imag); print(f"{n:6d} {z:12.4f} {m.zeros[n-1]:14.4f} {abs(z-m.zeros[n-1]):9.4f}")

  n     true zero    monad z[n-1]     error
    21      79.3374        81.1434    1.8060


   100     236.5242       236.8533    0.3290


  1000    1419.4225      1420.0974    0.6749


  5000    5447.8620      5448.2983    0.4363


 25000   21942.5924     21942.8748    0.2824
